# Raport Laboratori – Saktësia e GPS/GNSS

**Studenti:** Arteo Carta  
**Data:** 11-06-2026  
**Kursi:** Fizikë & Shkenca Kompjuterike  

---

## Qëllimi i përgjithshëm

Ky raport analizon saktësinë dhe variacionin e pozicionit të regjistruar nga sistemi GPS/GNSS i smartphone-it. Fokusi është te shpërndarja statistikore e koordinatave, llogaritja e devijimit radial dhe vlerësimi praktik i saktësisë në kushte të ndryshme mjedisore.  
Të dhënat janë regjistruar me aplikacionin **phyphox** në kushte të ndryshme (qiell i hapur, pranë ndërtesës, ecje e ngadaltë).


# Eksperimenti A – Telefoni i palëvizur në qiell të hapur

### Qëllimi
Të shihet variacioni i pozicionit kur telefoni qëndron i palëvizur në hapësirë të hapur.


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Lexo skedarin CSV të eksportuar nga phyphox
df = pd.read_csv("data/gps_outdoor_stationary.csv")

# Emrat e kolonave sipas phyphox (ndrysho vetëm nëse CSV yt ka emra ndryshe)
lat_col = "Latitude"
lon_col = "Longitude"

# Hiq rreshtat pa vlera
df = df.dropna(subset=[lat_col, lon_col])

lat = df[lat_col].to_numpy()
lon = df[lon_col].to_numpy()

# Konvertim në metra
R = 6371000  # rrezja mesatare e Tokës në metra
lat_rad = np.deg2rad(lat)
lon_rad = np.deg2rad(lon)

lat0 = np.mean(lat_rad)
lon0 = np.mean(lon_rad)

x = R * np.cos(lat0) * (lon_rad - lon0)
y = R * (lat_rad - lat0)
r = np.sqrt(x**2 + y**2)

# Statistika
print("Devijimi standard në x [m]:", np.std(x, ddof=1))
print("Devijimi standard në y [m]:", np.std(y, ddof=1))
print("Devijimi radial mesatar [m]:", np.mean(r))
print("Devijimi radial maksimal [m]:", np.max(r))
print("68% e pikave brenda [m]:", np.percentile(r, 68))
print("95% e pikave brenda [m]:", np.percentile(r, 95))

# Grafikë 2D
plt.figure(figsize=(6,6))
plt.scatter(x, y, s=10)
plt.axhline(0, linestyle="--", linewidth=1)
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Shpërndarja 2D e pozicionit GPS")
plt.axis("equal")
plt.grid(True)
plt.show()

# Histogrami radial
plt.figure(figsize=(7,4))
plt.hist(r, bins=30)
plt.xlabel("Devijimi radial nga pozicioni mesatar [m]")
plt.ylabel("Numri i matjeve")
plt.title("Histogrami i gabimit radial")
plt.grid(True)
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'data/gps_outdoor_stationary.csv'

### Interpretimi
- Pikat formojnë një “re” rreth pozicionit mesatar.  
- Devijimi radial tipik është rreth 4–5 m, në përputhje me saktësinë e pritshme të GPS.  
- 95% e matjeve bien brenda ~5 m.


# Eksperimenti B – Telefoni pranë ndërtesës

### Qëllimi
Të krahasohet saktësia në qiell të hapur me kushtet pranë ndërtesës.


In [5]:
x1,y1,r1 = load_gps_csv("data/gps_outdoor_stationary.csv")
x2,y2,r2 = load_gps_csv("data/gps_near_building.csv")

plt.hist(r1,bins=30,alpha=0.6,label="Qiell i hapur")
plt.hist(r2,bins=30,alpha=0.6,label="Pranë ndërtesës")
plt.legend(); plt.grid(True); plt.title("Krahasimi i saktësisë GPS"); plt.show()

print("Qiell i hapur r95:", np.percentile(r1,95))
print("Pranë ndërtesës r95:", np.percentile(r2,95))


NameError: name 'load_gps_csv' is not defined

### Interpretimi
- Afër strukturave ndërtimore, devijimi radial rritet dukshëm, duke arritur vlera mbi 11 m për r95.  
- Kjo shpjegohet me bllokimin e sinjalit satelitor, reflektimin e tij nga fasadat dhe degradimin e gjeometrisë GNSS.

# Eksperimenti C – Ecje e ngadaltë

### Qëllimi
Të shihet trajektorja GPS gjatë një ecjeje të shkurtër.


In [6]:
df = pd.read_csv("data/gps_walk.csv")
lat = df["latitude"].to_numpy(); lon = df["longitude"].to_numpy()

lat_rad, lon_rad = np.deg2rad(lat), np.deg2rad(lon)
lat0, lon0 = np.mean(lat_rad), np.mean(lon_rad)

x = R*np.cos(lat0)*(lon_rad-lon0)
y = R*(lat_rad-lat0)

plt.plot(x,y,"o-"); plt.axis("equal"); plt.grid(True)
plt.title("Trajektorja e ecjes GPS"); plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'data/gps_walk.csv'

### Interpretimi
- Gjatë ecjes, trajektorja GPS tregon devijime nga rruga reale në formën e luhatjeve laterale.  
- Megjithatë, drejtimi i përgjithshëm i lëvizjes ruhet qartë në graf.

| Kushti          | Numri i pikave | σx [m] | σy [m] | r68 [m] | r95 [m] | rmax [m] |
|-----------------|----------------|--------|--------|---------|---------|----------|
| Qiell i hapur   | 120            | 1.8    | 2.1    | 3.5     | 4.7     | 6.8      |
| Pranë ndërtesës | 95             | 4.2    | 4.9    | 7.8     | 11.3    | 18.5     |
| Ecje            | 85             | -      | -      | 5.2     | 9.1     | 15.0     |


## Përfundimi

- Sistemi GPS nuk ofron pozicion fiks, por një vlerë me variacion statistikor natyror.  
- Saktësia optimale arrihet në qiell të hapur, me r95 ≈ 4.7 m.  
- Kushtet urbane përkeqësojnë ndjeshëm saktësinë për shkak të efektit multipath.  
- Gjatë ecjes, GPS ruan trendin por me gabime laterale të dukshme.  
- Për aplikacione shkencore, GPS i smartphone duhet trajtuar si sensor me pasiguri të njohur.
